In [ ]:
# Install hypertools (dev-1.0 preview) -- run this first on Colab.
# On release this becomes: %pip install hypertools
%pip install -q "hypertools[interactive] @ git+https://github.com/ContextLab/hypertools.git@dev-1.0"
%pip install -q sentence-transformers

%matplotlib inline

# Five paintings, described in words, drawn in their own colors

This tutorial turns text into geometry, tinted by the art itself. Short descriptions of five famous paintings are embedded with a sentence-transformer and reduced *together* into one shared 3-D space. Each painting is one cloud, plotted in a color taken from the actual canvas; `animate='spin'` rotates the box while side panels list the descriptions.

The canvas color is k-means'd from the real image ([Wikimedia Commons](https://commons.wikimedia.org), cached); if the image can't be fetched, a hand-picked color is used. Text is embedded with a sentence-transformer, falling back to TF-IDF.

## 1. Imports, cache, and the bundled descriptions

In [2]:
import os, tempfile, textwrap, urllib.request
import numpy as np
from matplotlib.colors import to_rgb
from matplotlib.patches import Rectangle
import hypertools as hyp

CACHE = os.path.join(tempfile.gettempdir(), 'hypertools_tutorial')
os.makedirs(CACHE, exist_ok=True)
FILEPATH = 'https://commons.wikimedia.org/wiki/Special:FilePath/'

PAINTINGS = {
    'Starry Night': {'file': 'Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg',
        'fallback': '#2a3f6b',
        'blurb': 'Swirling cobalt sky, blazing yellow stars, a flame-like cypress.',
        'text': 'A turbulent night sky churns above a quiet village in thick swirling brushstrokes of deep cobalt and midnight blue. Great spirals of wind coil across the heavens and enormous yellow stars blaze like halos of fire over a small town and a dark flame-shaped cypress.'},
    'Mona Lisa': {'file': 'Mona_Lisa,_by_Leonardo_da_Vinci,_from_C2RMF_retouched.jpg',
        'fallback': '#6b5533',
        'blurb': 'A serene enigmatic smile; soft sfumato haze; misty blue mountains.',
        'text': 'A serene woman sits in three-quarter view with a faint enigmatic smile. Delicate sfumato dissolves every edge into soft brown and golden shadow, and behind her a dreamlike landscape of winding rivers and misty blue mountains recedes into golden light.'},
    'Water Lilies': {'file': 'Claude_Monet_-_Water_Lilies_-_1906,_Ryerson.jpg',
        'fallback': '#3f7d6e',
        'blurb': 'A still pond of floating lilies, reflected clouds, shimmering greens.',
        'text': 'The surface of a still pond fills the canvas with no horizon, only the skys reflection. Pale pink and white water lilies float across cool greens, blues and lavender while shimmering broken color captures reflected clouds and trembling summer light.'},
    'The Scream': {'file': 'Edvard_Munch,_1893,_The_Scream,_oil,_tempera_and_pastel_on_cardboard,_91_x_73_cm,_National_Gallery_of_Norway.jpg',
        'fallback': '#b5502e',
        'blurb': 'A figure clutching its face on a bridge; a blood-orange sky.',
        'text': 'A gaunt hairless figure stands on a bridge with hands clasped to its hollow face, mouth open in a silent scream. The sky burns in violent streaks of blood orange and fiery red above a swirling blue-black fjord, everything undulating with anguish.'},
    'The Great Wave': {'file': 'Tsunami_by_hokusai_19th_century.jpg',
        'fallback': '#1f4e79',
        'blurb': 'A towering wave clawing at tiny boats; tiny Mount Fuji far beyond.',
        'text': 'An enormous curling wave rears up over the sea, its crest breaking into countless clawing fingers of white foam over tiny fishing boats. Painted in deep Prussian blue and pale cream, the woodblock print frames the small cone of Mount Fuji far in the distance.'},
}
WINDOW, STEP = 10, 1

## 2. Embedding and canvas-color helpers

In [3]:
def embed(texts):
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('all-MiniLM-L6-v2')
        return np.asarray(model.encode(texts, show_progress_bar=False),
                          dtype=float)
    except Exception:
        from sklearn.feature_extraction.text import TfidfVectorizer
        vec = TfidfVectorizer(analyzer='char_wb',
                              ngram_range=(2, 4), min_df=1)
        return vec.fit_transform(texts).toarray().astype(float)


def windows(text, size, step):
    w = text.split()
    return [' '.join(w[i:i + size])
            for i in range(0, max(1, len(w) - size + 1), step)]


def canvas_color(spec):
    try:
        from PIL import Image
        from sklearn.cluster import KMeans
        dest = os.path.join(CACHE, 'paint_' + spec['file'][:20] + '.jpg')
        if not (os.path.exists(dest) and os.path.getsize(dest)):
            req = urllib.request.Request(
                FILEPATH + spec['file'] + '?width=400',
                headers={'User-Agent': 'ht-tutorial/1.0'})
            with urllib.request.urlopen(req, timeout=30) as r:
                open(dest, 'wb').write(r.read())
        im = Image.open(dest).convert('RGB')
        im.thumbnail((200, 200))
        px = np.asarray(im).reshape(-1, 3).astype(float)
        km = KMeans(n_clusters=6, n_init=4, random_state=0).fit(px)
        rgb = km.cluster_centers_[np.argmax(
            np.bincount(km.labels_, minlength=6))] / 255.0
        lum = 0.2126*rgb[0] + 0.7152*rgb[1] + 0.0722*rgb[2]
        if lum > 0.5:
            rgb = rgb * (0.5 / max(lum, 1e-6))
        return tuple(rgb)
    except Exception:
        return to_rgb(spec['fallback'])

## 3. Embed all paintings and reduce together

Every description's windows are embedded and reduced *together* (UMAP) so all five clouds share one space, then we trim each cloud's few farthest outliers so the clouds read cleanly.

The UMAP kwargs trade global structure for tighter per-painting clumps. **`n_neighbors=12`** sets how much of the graph each point sees: small values (as here) preserve local neighborhoods, so the windows of one description stay together, while large values would pull the layout toward the global average and blur the five clouds into each other. **`min_dist=0.25`** is small, which lets points in a clump pack closely instead of being pushed apart, making each painting a compact cloud. **`random_state=42`** fixes UMAP's stochastic optimization so the layout is reproducible.

In [4]:
# embed ALL windows in one call so every vector shares the same
# dimensionality (the TF-IDF fallback fits its vocabulary once)
all_windows, owners, colors_by_name = [], [], {}
for name, spec in PAINTINGS.items():
    wins = windows(spec['text'], WINDOW, STEP)
    all_windows += wins
    owners += [name] * len(wins)
    colors_by_name[name] = canvas_color(spec)
all_vecs = embed(all_windows)
owners = np.array(owners)

import warnings
warnings.filterwarnings('ignore',
                        message=r'n_jobs value \d+ overridden.*',
                        category=UserWarning)  # benign UMAP/random_state note

red = np.asarray(hyp.reduce(
    all_vecs, reduce={'model': 'UMAP',
                      'kwargs': {'n_neighbors': 12, 'min_dist': 0.25,
                                 'random_state': 42}}, ndims=3))
# no rescaling here: hyp.plot already mean-centers every dataset and rescales
# them into [-1, 1] with ONE shared affine before drawing (and the outlier
# trim below compares distances to a percentile, so it is scale-free)
keep = np.ones(len(red), bool)
for name in PAINTINGS:
    idx = np.where(owners == name)[0]
    dist = np.linalg.norm(red[idx] - np.median(red[idx], 0), axis=1)
    keep[idx[dist > np.percentile(dist, 85)]] = False
red, owners = red[keep], owners[keep]

clouds = [red[owners == name] for name in PAINTINGS]
colors = [colors_by_name[name] for name in PAINTINGS]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 4. Plot: one cloud/color per painting, spinning, with side panels

`reduce=None` tells `hyp.plot` the clouds are already 3-D, so it skips its default IncrementalPCA and plots the coordinates we produced above. `color=colors` gives one canvas color per cloud: the color list lines up element-for-element with the list of clouds, so painting *i* is drawn in painting *i*'s color. `animate='spin'` orbits the camera `rotations=2` full turns over `duration=12` seconds at `frame_rate=20` fps (240 frames) while the points themselves stay fixed, so the parallax of the turning view is what reveals the 3-D layout of the clouds.

In [5]:
duration, fps = 12, 20
fig, ani = hyp.plot(clouds, '.', color=colors, reduce=None,
                    markersize=5, animate='spin', rotations=2,
                    duration=duration, frame_rate=fps,
                    size=(11, 6.6), show=False)

ax = fig.axes[0]
ax.set_position([0.0, 0.0, 0.6, 1.0])
fig.text(0.30, 0.965,
         'Five paintings, described in words, drawn in their own colors',
         ha='center', va='top', fontsize=13.5, fontweight='bold',
         color='#1a1a1a')
for i, name in enumerate(PAINTINGS):
    y = 0.90 - i * 0.178
    c = colors_by_name[name]
    fig.text(0.635, y, name, ha='left', va='top', fontsize=13,
             fontweight='bold', color=c)
    body = '\n'.join(textwrap.wrap(PAINTINGS[name]['blurb'], 42)[:3])
    fig.text(0.635, y - 0.032, body, ha='left', va='top',
             fontsize=8.8, color=c)
    fig.add_artist(Rectangle((0.635, y - 0.11), 0.12, 0.014,
                             transform=fig.transFigure, facecolor=c,
                             edgecolor='#dddddd', lw=0.4))

## 5. Display the animation

In [6]:
fig.set_dpi(100)  # halve hypertools' default 200-dpi canvas for a lighter GIF
ani.save('painting_embeddings.gif', fps=fps)
print('saved painting_embeddings.gif')

saved painting_embeddings.gif


![five paintings, described in words, drawn in their own colors](painting_embeddings.gif)